In [1]:
import pandas as pd

In [2]:
import sqlite3

In [3]:
df = pd.read_csv("News Articles.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   headlines    10000 non-null  object
 1   description  10000 non-null  object
 2   content      10000 non-null  object
 3   url          10000 non-null  object
 4   category     10000 non-null  object
dtypes: object(5)
memory usage: 390.8+ KB


In [5]:
df.head(5)

,headlines,description,content,url,category
0,Nirmala Sitharaman to equal Morarji Desai’s re...,With the presentation of the interim budget on...,"Sitharaman, the first full-time woman finance ...",https://indianexpress.com/article/business/bud...,business
1,"‘Will densify network, want to be at least no....","'In terms of market share, we aim to double it...",The merger of Tata group’s budget airlines Air...,https://indianexpress.com/article/business/avi...,business
2,Air India group to induct an aircraft every si...,Air India currently has 117 operational aircra...,The Air India group plans to induct one aircra...,https://indianexpress.com/article/business/avi...,business
3,Red Sea woes: Exporters seek increased credit ...,Rising attacks forced shippers to consider the...,Indian exporters have asked the central govern...,https://indianexpress.com/article/business/red...,business
4,Air India group to induct a plane every 6 days...,"Apart from fleet expansion, 2024 will also see...",The Air India group plans to induct one aircra...,https://indianexpress.com/article/business/avi...,business


# Store in SQL

In [6]:
conn = sqlite3.connect("news.db")
df.to_sql("news_articles", conn, if_exists='replace', index=False)

print("New data saved to SQLite successfully.")

New data saved to SQLite successfully.


In [11]:
print(pd.read_sql("SELECT COUNT(*) FROM news_articles", conn))

   COUNT(*)
0     10000


In [10]:
conn = sqlite3.connect("news.db")
df = pd.read_sql("SELECT * FROM news_articles LIMIT 5", conn)
df

,headlines,description,content,url,category
0,Nirmala Sitharaman to equal Morarji Desai’s re...,With the presentation of the interim budget on...,"Sitharaman, the first full-time woman finance ...",https://indianexpress.com/article/business/bud...,business
1,"‘Will densify network, want to be at least no....","'In terms of market share, we aim to double it...",The merger of Tata group’s budget airlines Air...,https://indianexpress.com/article/business/avi...,business
2,Air India group to induct an aircraft every si...,Air India currently has 117 operational aircra...,The Air India group plans to induct one aircra...,https://indianexpress.com/article/business/avi...,business
3,Red Sea woes: Exporters seek increased credit ...,Rising attacks forced shippers to consider the...,Indian exporters have asked the central govern...,https://indianexpress.com/article/business/red...,business
4,Air India group to induct a plane every 6 days...,"Apart from fleet expansion, 2024 will also see...",The Air India group plans to induct one aircra...,https://indianexpress.com/article/business/avi...,business


## Search by Keyword in headlines

In [12]:
keyword = "Air India"
df = pd.read_sql(f"""
    SELECT * FROM news_articles
    WHERE headlines LIKE '%{keyword}%'
""", conn)

df.head()

,headlines,description,content,url,category
0,"‘Will densify network, want to be at least no....","'In terms of market share, we aim to double it...",The merger of Tata group’s budget airlines Air...,https://indianexpress.com/article/business/avi...,business
1,Air India group to induct an aircraft every si...,Air India currently has 117 operational aircra...,The Air India group plans to induct one aircra...,https://indianexpress.com/article/business/avi...,business
2,Air India group to induct a plane every 6 days...,"Apart from fleet expansion, 2024 will also see...",The Air India group plans to induct one aircra...,https://indianexpress.com/article/business/avi...,business
3,DGCA slaps Rs 1.1 crore penalty on Air India o...,Sources in the DGCA indicated that the action ...,Aviation safety regulator Directorate General ...,https://indianexpress.com/article/business/avi...,business
4,"Air India’s first A350 enters service, carries...","According to Airbus, in a standard three-class...","In a long-bygone era, Air India helmed by JRD ...",https://indianexpress.com/article/business/avi...,business


## Search with Multiple Conditions

In [13]:
df = pd.read_sql("""
    SELECT * FROM news_articles
    WHERE category = 'technology '
      AND headlines LIKE '%Air India%'
""", conn)

df.head()

,headlines,description,content,url,category


## Count Results

In [14]:
df = pd.read_sql("""
    SELECT COUNT(*) AS total
    FROM news_articles
    WHERE category = 'business'
""", conn)

print(df)

   total
0   2000


In [15]:
conn.close()

# Store in ElasticSearch

In [16]:
# !pip install elasticsearch

In [17]:
from elasticsearch import Elasticsearch

In [18]:
es = Elasticsearch("http://localhost:9200")

# Test connection
print(es.info())

{'name': 'd17b0c6368b7', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'uy2QIg5CS0OJ81JoX1ohIQ', 'version': {'number': '8.13.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '09df99393193b2c53d92899662a8b8b3c55b45cd', 'build_date': '2024-03-22T03:35:46.757803203Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [19]:
index_name = "news-index"

# Create a new index if needed (optional step)
if not es.indices.exists(index=index_name):
    es.indices.create(index=index_name)

# Upload each row
for i, row in df.iterrows():
    es.index(index=index_name, document=row.to_dict())

print(f"{len(df)} documents indexed to ElasticSearch.")

1 documents indexed to ElasticSearch.


In [20]:
# Confirm ElasticSearch
res = es.count(index="news-index")
print("ElasticSearch count:", res['count'])

ElasticSearch count: 10000


In [21]:
sample = es.search(index="news-index", query={"match_all": {}}, size=1)
print(sample["hits"]["hits"][0]["_source"].keys())


dict_keys(['headlines', 'description', 'content', 'url', 'category'])


## Search for a key word: 'Trump'

In [22]:
query = {
    "size": 5,
    "query": {
        "match": {
            "headlines": "Trump"
        }
    }
}

res = es.search(index="news-index", body=query)
for hit in res["hits"]["hits"]:
    print(hit["_source"]["headlines"])


Sushmita Sen says Donald Trump was never her ‘boss’ on Miss Universe, working with him wasn’t ‘easy or fun’


## Search phrase in description

In [23]:
query = {
    "size": 5,
    "query": {
        "match_phrase": {
            "description": "climate change"
        }
    }
}

res = es.search(index="news-index", body=query)
for hit in res["hits"]["hits"]:
    print(hit["_source"]["description"])

A new study found that climate change could potentially make diarrhoeal illnesses more common.
Members of the University of Copenhagen, Denmark, visited Delhi University and discussed climate change and other issues.
Takakia, a genus of moss that has survived for millions of years, is now threatened by climate change.
Can a sun shade tethered to an asteroid be a practical solution in the fight against climate change?
Pope Francis warns the world is "collapsing" due to climate change and may be "near its breaking point."


In [24]:
query = {
    "size": 5,
    "query": {
        "multi_match": {
            "query": "Donald Trump",
            "fields": ["headlines", "description", "content"]
        }
    }
}

res = es.search(index="news-index", body=query)
for hit in res["hits"]["hits"]:
    print(hit["_source"]["headlines"])

Sushmita Sen says Donald Trump was never her ‘boss’ on Miss Universe, working with him wasn’t ‘easy or fun’
New York City bans TikTok on government-owned devices over security concerns
Sushmita Sen reveals how she paid the bills when roles dried up, says she reached out to streamers for work when she was ‘jobless’
Gutsy bouncer-happy Neil Wagner wins a thriller for New Zealand against England
ONGC hopes to recover over $500 million dividend as sanctions on Venezuela eased


## Filter by category

In [25]:
query = {
    "size": 5,
    "query": {
        "term": {
            "category.keyword": "business"
        }
    }
}

res = es.search(index="news-index", body=query)
for hit in res["hits"]["hits"]:
    print(hit["_source"]["headlines"], "|", hit["_source"]["category"])

Tackling counterfeiting, tax evasion key to safeguarding economic stability: CBIC chief | business
Centre says ready to bring 28% online gaming GST from Oct 1; all states yet to pass laws | business
Air India frontline staff to soon have new uniforms; airline partners with fashion designer Manish Malhotra | business
Ashwin Dani, former Asian Paints chair, dies at 79 | business
OMCs incurring under-recoveries of over Rs 7/litre on petrol, diesel sales, says Nomura | business


In [26]:
query = {
    "size": 5,
    "query": {
        "term": {
            "category.keyword": "education"
        }
    }
}

res = es.search(index="news-index", body=query)
for hit in res["hits"]["hits"]:
    print(hit["_source"]["headlines"], "|", hit["_source"]["category"])

MIT faculty, classes in workshop: IIT Kanpur alumnus from first batch shares his experience | education
CTET August 2023 answer key released; challenge window open till Sept 18 | education
IIT Madras Pravartak partners with Simplilearn to train students on digital skills | education
IIT Madras launches Jal Dhan Campaign to conserve water | education
Telangana CM inaugurates 9 new govt medical colleges; 8 more to come up in 2024 | education


## Search inside content field

In [27]:
query = {
    "size": 5,
    "query": {
        "match": {
            "content": "Sitharaman"
        }
    }
}

res = es.search(index="news-index", body=query)
for hit in res["hits"]["hits"]:
    print(hit["_source"]["headlines"])

Global economy suffering the brunt of simultaneous wars: Nirmala Sitharaman
FM Nirmala Sitharaman asks financial entities to ensure customers nominate heirs
India fastest-growing economy in world today: FM Nirmala Sitharaman during No-Confidence Motion debate
Any border tax by developed nations to meet green commitments is morally wrong: FM
FM on conflict in West Asia: Concerns back on fuel, inflation
